# 01 — Data Quality & Readiness

## tl;dr

The synthetic source contains **40 suppliers, 90 unique parts, and 1,800 purchase-order lines**. Required primary keys are unique, mandatory analytical fields are populated, PO quantities and defect counts pass validity checks, and every PO maps to a known supplier and part. The dataset is ready for the portfolio analysis, with the important caveat that synthetic completeness does not represent the messiness of a production ERP system.

## Context & Methods

Before supplier scores can support a decision, we need to establish whether the underlying records are complete, unique, valid, and joinable.

### Key Assumptions

- Each purchase-order row represents one received PO line.
- `supplier_id`, `po_id`, and inventory `part_id` are expected to be unique at their stated grain.
- Dates use a common calendar and quantities are expressed in units.
- The generated data is a controlled demonstration, not a live ERP extract.

### 1. Load the source tables

In [1]:
from pathlib import Path
import pandas as pd

from src.supply_chain_risk.analytics import RISK_WEIGHTS, apply_weight_scenario, build_analysis, load_data

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
tables, supplier_scores, part_risk, kpis = build_analysis(DATA_DIR)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

{name: frame.shape for name, frame in tables.items()}

{'suppliers': (40, 7), 'parts': (160, 6), 'purchase_orders': (1800, 10), 'inventory': (90, 4)}

### 2. Inspect a bounded record sample

In [2]:
tables['purchase_orders'].head(8)

,po_id,supplier_id,part_id,order_date,promised_date,receipt_date,quantity,unit_price,defect_quantity,status
0,PO-000001,SUP-039,PRT-0087,2026-01-09,2026-03-06,2026-03-05,370,157.62,5,Received
1,PO-000002,SUP-003,PRT-0081,2025-10-28,2026-01-04,2025-12-30,93,106.34,0,Received
2,PO-000003,SUP-036,PRT-0043,2025-12-21,2026-03-09,2026-03-09,137,41.28,1,Received
3,PO-000004,SUP-024,PRT-0025,2025-10-28,2025-12-23,2025-12-23,373,602.91,8,Received
4,PO-000005,SUP-007,PRT-0036,2026-06-04,2026-07-22,2026-08-02,319,174.22,4,Received
5,PO-000006,SUP-023,PRT-0039,2026-03-25,2026-05-22,2026-05-14,210,573.58,1,Received
6,PO-000007,SUP-024,PRT-0058,2026-07-01,2026-08-03,2026-08-02,59,289.08,0,Received
7,PO-000008,SUP-014,PRT-0022,2026-04-27,2026-06-12,2026-06-06,34,70.53,2,Received


### 3. Test completeness, uniqueness, validity, and joins

In [3]:
quality_rows = []
for name, frame in tables.items():
    quality_rows.append({"table": name, "rows": len(frame), "columns": len(frame.columns), "missing_required_values": int(frame.isna().sum().sum())})
quality_summary = pd.DataFrame(quality_rows)
quality_summary["duplicate_primary_keys"] = [
    tables["suppliers"].duplicated("supplier_id").sum(),
    tables["parts"].duplicated(["part_id", "supplier_id"]).sum(),
    tables["purchase_orders"].duplicated("po_id").sum(),
    tables["inventory"].duplicated("part_id").sum(),
]
quality_summary["status"] = quality_summary.apply(lambda row: "Pass" if row["missing_required_values"] == 0 and row["duplicate_primary_keys"] == 0 else "Review", axis=1)
quality_summary

,table,rows,columns,missing_required_values,duplicate_primary_keys,status
0,suppliers,40,7,0,0,Pass
1,parts,160,6,0,0,Pass
2,purchase_orders,1800,10,0,0,Pass
3,inventory,90,4,0,0,Pass


### 4. Verify relational coverage and business rules

In [4]:
po = tables["purchase_orders"]
checks = pd.DataFrame([
    {"check": "POs with unknown supplier", "exceptions": int((~po["supplier_id"].isin(set(tables["suppliers"]["supplier_id"]))).sum())},
    {"check": "POs with unknown part", "exceptions": int((~po["part_id"].isin(set(tables["parts"]["part_id"]))).sum())},
    {"check": "Non-positive PO quantities", "exceptions": int((po["quantity"] <= 0).sum())},
    {"check": "Defects exceeding quantity", "exceptions": int((po["defect_quantity"] > po["quantity"]).sum())},
    {"check": "Receipt before order date", "exceptions": int((po["receipt_date"] < po["order_date"]).sum())},
])
checks["status"] = checks["exceptions"].map(lambda value: "Pass" if value == 0 else "Review")
checks

,check,exceptions,status
0,POs with unknown supplier,0,Pass
1,POs with unknown part,0,Pass
2,Non-positive PO quantities,0,Pass
3,Defects exceeding quantity,0,Pass
4,Receipt before order date,0,Pass


## Results

The controlled dataset passes the checks needed for the current analysis. The clean result is expected because the generator deliberately enforces valid keys and ranges. In production, the same checks would produce an exception table for remediation rather than silently dropping records.

## Takeaways

- Table grains and keys are explicit and testable.
- Supplier and part joins preserve the intended population.
- Management metrics are calculated only after validation.
- A real implementation would add data owners, refresh checks, rejected-row logging, currency normalization, and ERP reconciliation.